# 03 — EDA: physicochemical modeling data (TEM-1 / Firnberg)

Exploratory analysis of the **physicochemical** feature matrix built in `03a`. This is the data the
traditional-ML benchmark (notebook `04`) trains on: for each single missense variant we describe the
substitution only by its **physical chemistry** — change in hydrophobicity, charge, side-chain volume,
polarity, flexibility, and the standard multidimensional property scales — with no amino-acid or
position identity.

The question this notebook answers before any modeling: **how much signal about functionality
(`DMS_score_bin`) is carried by physical properties alone**, which properties carry it, and how
redundant the descriptor set is. Companion to notebook `01` (the AA-identity EDA), restricted to the
physicochemical columns.

In [1]:
# self-contained: resolve project root via .projectroot, read from disk
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root))
from paths import *

import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

# project palette: blues, greens, purples, dark pinks only
PAL = {"blue":"#2c6fbb", "green":"#2e8b57", "purple":"#7a4fa3", "pink":"#b03060"}
sns.set_theme(style="whitegrid")
FIGDIR = FIGURES / "03_EDA_physicochemical"; FIGDIR.mkdir(parents=True, exist_ok=True)
print("root:", root, "| fig dir:", FIGDIR.relative_to(BASE_DIR))

root: /Users/kdave2/Beta-Lactamase Mutagenesis/1 - ML | fig dir: results/figures/03_EDA_physicochemical


In [2]:
# get the data + feature metadata (which columns are which family)
PCDIR = PROCESSED / "physicochemical"
df   = pd.read_parquet(PCDIR / "modeling_dataset.parquet")
meta = json.load(open(PCDIR / "feature_metadata.json"))
feat_cols = meta["feature_columns"]; label_col = "DMS_score_bin"; score_col = "DMS_score"
# validate the boundary (CLAUDE.md): labels present, no NaN in features
assert df[label_col].isin([0,1]).all()
assert not df[feat_cols].isna().any().any()
print("rows, columns:", df.shape, "| features:", len(feat_cols))
print("families:", {k: sum(1 for c in feat_cols if meta['groups'][c]==k) for k in ['interpretable','qsar_scale','flag']})

rows, columns: (4783, 368) | features: 362
families: {'interpretable': 42, 'qsar_scale': 306, 'flag': 14}


## The label: functional vs non-functional

`DMS_score_bin` is the binary target the benchmark predicts. **Positive class = functional (1).** Same
label as the AA-identity table (03a reuses it), so the class balance is identical — repeated here so
this notebook stands alone.

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vc = df[label_col].value_counts().sort_index()
ax[0].bar(["non-functional (0)","functional (1)"], vc.values,
          color=[PAL["pink"], PAL["green"]], edgecolor="black", linewidth=.6)
ax[0].set_title(f"Class balance (n={len(df)})"); ax[0].set_ylabel("variants")
for i,v in enumerate(vc.values): ax[0].text(i, v+20, str(v), ha="center")
sns.histplot(df[score_col], bins=40, ax=ax[1], color=PAL["blue"])
ax[1].set_title("DMS score — note spikes at the extremes"); ax[1].set_xlabel("DMS score")
fig.tight_layout(); fig.savefig(FIGDIR/"label_distribution.pdf", bbox_inches="tight"); plt.show()
print("class balance:", vc.to_dict())

class balance: {0: 2386, 1: 2397}


## Do the headline properties separate the classes?

Six interpretable substitution properties a biochemist would name first — change in hydropathy, charge,
side-chain volume, polarity, flexibility, and the Chou-Fasman α-helix propensity. For each, the
distribution of the **delta (mut − wt)** is shown split by functional class. Separation between the two
colours is signal the model can use; heavy overlap is not.

In [4]:
head = [("hydropathy_kd_delta","Δ hydropathy (Kyte-Doolittle)"),
        ("charge_delta","Δ formal charge"),
        ("volume_delta","Δ side-chain volume"),
        ("polarity_grantham_delta","Δ polarity (Grantham)"),
        ("flexibility_delta","Δ flexibility (B-P)"),
        ("cf_alpha_delta","Δ Chou-Fasman α")]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax,(col,title) in zip(axes.ravel(), head):
    for cls,color,lab in [(0,PAL["pink"],"non-func"),(1,PAL["green"],"func")]:
        sns.kdeplot(df.loc[df[label_col]==cls, col], ax=ax, color=color, fill=True, alpha=.35, label=lab, lw=1.4)
    ax.set_title(title); ax.set_xlabel(""); ax.legend(fontsize=8)
fig.suptitle("Property change distributions by functional class", y=1.01, fontsize=13)
fig.tight_layout(); fig.savefig(FIGDIR/"property_distributions_by_class.pdf", bbox_inches="tight"); plt.show()

## Functional rate across a property gradient

Binning a continuous property and plotting P(functional) per bin shows whether the relationship is
monotonic (a model can lean on it as a trend) or non-linear (the model must learn a shape). Shown for
the change in hydropathy and the change in side-chain volume — two of the strongest single properties.

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a,(col,title,color) in zip(ax, [("hydropathy_kd_delta","Δ hydropathy",PAL["blue"]),
                                    ("volume_delta","Δ side-chain volume",PAL["purple"])]):
    b = pd.qcut(df[col], 12, duplicates="drop")
    rate = df.groupby(b, observed=True)[label_col].mean()
    centers = [iv.mid for iv in rate.index]
    a.plot(centers, rate.values, "-o", color=color, lw=1.6)
    a.axhline(df[label_col].mean(), color=PAL["pink"], ls="--", lw=1, label=f"overall {df[label_col].mean():.2f}")
    a.set_title(f"Functional rate vs {title}"); a.set_xlabel(title); a.set_ylabel("P(functional)"); a.legend()
fig.tight_layout(); fig.savefig(FIGDIR/"functional_rate_by_property.pdf", bbox_inches="tight"); plt.show()

## Which properties carry the most signal?

Physicochemical features are continuous, so we test **association with the binary label** the right way
for each: point-biserial correlation (linear), and the rank-based Mann-Whitney effect size (AUC of the
property as a one-feature classifier, |0.5−AUC| as effect). Top features by |point-biserial r| are
reported and written to a table. This is the physicochemical analogue of notebook 01's chi-square /
Cramér's V association tests.

In [6]:
rows = []
y = df[label_col].values
for c in feat_cols:
    x = df[c].values.astype(float)
    if x.std() == 0:
        continue
    r, p = stats.pointbiserialr(y, x)
    try:
        auc = roc_auc = stats.mannwhitneyu(x[y==1], x[y==0], alternative="two-sided").statistic / ((y==1).sum()*(y==0).sum())
    except ValueError:
        auc = 0.5
    rows.append({"feature":c, "family":meta["subfamilies"][c], "kind":meta["kind"][c],
                 "pointbiserial_r":r, "abs_r":abs(r), "mannwhitney_auc":auc, "p_value":p})
assoc = pd.DataFrame(rows).sort_values("abs_r", ascending=False).reset_index(drop=True)
TABLES.mkdir(parents=True, exist_ok=True)
assoc.round(4).to_csv(TABLES/"03_EDA_physicochemical_association.csv", index=False)
print("top 12 physicochemical features by |point-biserial r|:")
print(assoc.head(12)[["feature","family","kind","pointbiserial_r","mannwhitney_auc","p_value"]].to_string(index=False))

top 12 physicochemical features by |point-biserial r|:
                feature            family  kind  pointbiserial_r  mannwhitney_auc      p_value
    hydropathy_kd_delta     hydropathy_kd delta         0.219516         0.625113 2.795812e-53
         desc_AF1_delta                AF delta        -0.213927         0.378946 1.247875e-50
            desc_KF4_wt                KF    wt         0.211617         0.622592 1.478253e-49
   polarity_grantham_wt polarity_grantham    wt         0.204142         0.614700 3.611698e-46
          desc_VHSE1_wt              VHSE    wt        -0.202802         0.380602 1.416353e-45
            desc_AF1_wt                AF    wt         0.199758         0.613441 3.045180e-44
        desc_BLOSUM1_wt            BLOSUM    wt         0.194503         0.601351 5.403906e-42
polarity_grantham_delta polarity_grantham delta        -0.193522         0.393781 1.398029e-41
     desc_BLOSUM1_delta            BLOSUM delta        -0.191243         0.396365 1.247241

In [7]:
# top-15 features as a horizontal bar of |r|, colored by family kind
top = assoc.head(15)[::-1]
kind_color = {"wt":PAL["blue"], "mut":PAL["green"], "delta":PAL["purple"], "flag":PAL["pink"]}
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.barh(range(len(top)), top.abs_r.values,
        color=[kind_color.get(k, PAL["blue"]) for k in top["kind"]], edgecolor="black", linewidth=.4)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.feature, fontsize=8)
ax.set_xlabel("|point-biserial r| with functional label")
ax.set_title("Strongest physicochemical associations with functionality")
handles = [plt.Rectangle((0,0),1,1,color=kind_color[k]) for k in ["wt","mut","delta","flag"]]
ax.legend(handles, ["wt value","mut value","delta","flag"], fontsize=8, title="feature kind")
fig.tight_layout(); fig.savefig(FIGDIR/"top_associations.pdf", bbox_inches="tight"); plt.show()
print("strongest single property |r| =", round(assoc.abs_r.max(),3),
      "-- same moderate ceiling as the AA-identity Cramer's V (~0.2-0.33)")

strongest single property |r| = 0.22 -- same moderate ceiling as the AA-identity Cramer's V (~0.2-0.33)


## How redundant is the descriptor set?

362 descriptors sound like a lot, but the QSAR scales are known to be correlated (many encode
hydrophobicity or size in different bases). The correlation heatmap over the interpretable-delta block
plus a few representative QSAR-scale deltas shows how much of the matrix is repeating the same physical
axes — which is why the linear model does nearly as well as the trees in notebook 04, and why raw
feature count is not the same as independent signal.

In [8]:
rep = [c for c in feat_cols if c.endswith("_delta") and (
        meta["groups"][c]=="interpretable" or meta["subfamilies"][c] in {"Z","VHSE","T","KF"})]
rep = rep[:26]   # keep the heatmap legible
corr = df[rep].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="PuOr", vmin=-1, vmax=1, center=0, square=True, linewidths=.3,
            cbar_kws={"label":"Pearson r"}, ax=ax,
            xticklabels=[c.replace("_delta","") for c in rep],
            yticklabels=[c.replace("_delta","") for c in rep])
ax.set_title("Redundancy among physicochemical deltas (interpretable + representative QSAR scales)")
fig.tight_layout(); fig.savefig(FIGDIR/"feature_correlation.pdf", bbox_inches="tight"); plt.show()
off = corr.where(~np.eye(len(corr), dtype=bool))
hi = (off.abs() > 0.7).sum().sum() / 2
print(f"pairs with |r|>0.7 among {len(rep)} representative deltas: {int(hi)} -- descriptor set is substantially redundant")

pairs with |r|>0.7 among 26 representative deltas: 19 -- descriptor set is substantially redundant


## Leakage check for the downstream benchmark

The benchmark (notebook `04`) standardizes and feeds these columns directly. Confirm here that no
physicochemical feature is a deterministic function of the label — nothing tracks `DMS_score_bin`
perfectly — which would be leakage-by-construction. (03a already asserts this at save time; repeated
here as the EDA's own gate.)

In [9]:
Xv = df[feat_cols].values.astype(float)
active = np.where(Xv.std(0) > 0)[0]
mx = max(abs(np.corrcoef(Xv[:,j], y)[0,1]) for j in active)
assert mx < 0.999, "leakage: a feature perfectly tracks the label"
assert not any(any(t in c.lower() for t in ["dms","score","bin","label"]) for c in feat_cols)
print(f"max |corr(feature, y)| = {mx:.3f}  -> no leakage; safe to model in notebook 04")

max |corr(feature, y)| = 0.220  -> no leakage; safe to model in notebook 04


## Summary

- **Balanced binary target** (~50/50 across 4,783 single missense variants) — identical to the
  AA-identity table by construction.
- **Physical properties carry real but moderate signal**: the strongest single descriptor reaches
  |point-biserial r| ≈ 0.2, the same moderate ceiling as the strongest identity feature (Cramér's V
  ≈ 0.2–0.33 in notebook 01). Change in hydropathy, side-chain volume, and polarity separate the
  classes most; charge change and the QSAR hydrophobicity/size axes echo them.
- **The descriptor set is substantially redundant** — many QSAR scales re-encode the same physical
  axes — so 362 columns is far fewer than 362 independent signals, foreshadowing the linear model's
  near-parity with the trees in the benchmark.
- **No leakage**: no property is a label proxy. The matrix is model-ready; notebook `04` measures how
  far this position-free physical-chemistry representation actually gets across the three split schemes.